# SQL Fundamentals with DuckDB (Solution)

Reference solutions for `exercise_8_sql_with_duckdb.ipynb`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import duckdb

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.append(str(project_root))

from src.utilities.project_paths import RAW_DIR

DTA_DIR = RAW_DIR  / 'TZNPS5_20_11_STATA'

COLS_HH = ['interview__id', 't0_region', 't0_district',
           'nps4_hhsize', 'nps4_nplots', 'int_result']

households = pd.read_stata(
    DTA_DIR / 'TZNPS5_20.dta',
    columns=COLS_HH,
    convert_categoricals=False,
)
roster = pd.read_stata(
    DTA_DIR / 't2_roster.dta',
    columns=['interview__id', 'Calc_Age'],
    convert_categoricals=False,
).dropna()

duckdb.register('households', households)
duckdb.register('roster', roster)

print(f'households : {len(households):,} rows')
print(f'roster     : {len(roster):,} rows')
print()
print('households columns:', list(households.columns))

households : 791 rows
roster     : 3,510 rows

households columns: ['interview__id', 't0_region', 't0_district', 'nps4_hhsize', 'nps4_nplots', 'int_result']


---
# Part A — Looking at the Data

| Clause | What it does |
|---|---|
| `SELECT *` | Return all columns |
| `SELECT col1, col2` | Return specific columns |
| `LIMIT n` | Return only the first n rows |

## A1. How many rows are there?

In [2]:
duckdb.sql("""
    SELECT COUNT(*) AS n_rows
    FROM households
""").to_df()

,n_rows
0,791


## A2. Peek at the data

In [3]:
duckdb.sql("""
    SELECT *
    FROM households
    LIMIT 5
""").to_df()

,interview__id,t0_region,t0_district,nps4_hhsize,nps4_nplots,int_result
0,29b1d1314dcd4092bf2461b52b444050,16.0,166.0,1.0,2.0,1.0
1,a5d01f703a614cb494771ee875f969c6,16.0,166.0,3.0,4.0,1.0
2,a20b7d8c12c5479c80fc5ed63b981027,7.0,73.0,3.0,NaN,1.0
3,7aa2f3cef525481f8f4648ad244da415,18.0,181.0,6.0,2.0,1.0
4,0bf11205744c45f7929c37b4ca305a1b,18.0,181.0,2.0,2.0,1.0


## A3. Select specific columns

In [4]:
duckdb.sql("""
    SELECT interview__id, t0_region, nps4_hhsize
    FROM households
    LIMIT 10
""").to_df()

,interview__id,t0_region,nps4_hhsize
0,29b1d1314dcd4092bf2461b52b444050,16.0,1.0
1,a5d01f703a614cb494771ee875f969c6,16.0,3.0
2,a20b7d8c12c5479c80fc5ed63b981027,7.0,3.0
3,7aa2f3cef525481f8f4648ad244da415,18.0,6.0
4,0bf11205744c45f7929c37b4ca305a1b,18.0,2.0
5,59ad078d29874845941ce9983343dbb3,16.0,NaN
6,2539b62f6d5a4e7080df96ce76b6a2eb,16.0,9.0
7,500d2127ecd64d40b47bb2f2365b4629,18.0,NaN
8,23f388799563482ea3067cf6289386dd,16.0,NaN
9,9b1bdc92507e495693b487349c520fa2,18.0,7.0


---
# Part B — Filtering with WHERE

| Operator | Meaning | Example |
|---|---|---|
| `=` | equal | `int_result = 1` |
| `>` / `<` | greater / less than | `nps4_hhsize > 5` |
| `>=` / `<=` | greater or equal / less or equal | `nps4_hhsize >= 3` |
| `AND` | both conditions must be true | `int_result = 1 AND nps4_hhsize > 5` |
| `OR` | at least one must be true | `t0_region = 1 OR t0_region = 2` |

## B1. Completed interviews only

In [5]:
duckdb.sql("""
    SELECT *
    FROM households
    WHERE int_result = 1
    LIMIT 10
""").to_df()

,interview__id,t0_region,t0_district,nps4_hhsize,nps4_nplots,int_result
0,29b1d1314dcd4092bf2461b52b444050,16.0,166.0,1.0,2.0,1.0
1,a5d01f703a614cb494771ee875f969c6,16.0,166.0,3.0,4.0,1.0
2,a20b7d8c12c5479c80fc5ed63b981027,7.0,73.0,3.0,NaN,1.0
3,7aa2f3cef525481f8f4648ad244da415,18.0,181.0,6.0,2.0,1.0
4,0bf11205744c45f7929c37b4ca305a1b,18.0,181.0,2.0,2.0,1.0
5,59ad078d29874845941ce9983343dbb3,16.0,166.0,NaN,NaN,1.0
6,2539b62f6d5a4e7080df96ce76b6a2eb,16.0,166.0,9.0,2.0,1.0
7,500d2127ecd64d40b47bb2f2365b4629,18.0,184.0,NaN,NaN,1.0
8,23f388799563482ea3067cf6289386dd,16.0,166.0,NaN,NaN,1.0
9,9b1bdc92507e495693b487349c520fa2,18.0,181.0,7.0,1.0,1.0


## B2. Large households

In [6]:
duckdb.sql("""
    SELECT interview__id, t0_region, nps4_hhsize
    FROM households
    WHERE nps4_hhsize > 6
""").to_df()

,interview__id,t0_region,nps4_hhsize
0,2539b62f6d5a4e7080df96ce76b6a2eb,16.0,9.0
1,9b1bdc92507e495693b487349c520fa2,18.0,7.0
2,4881e03739c2482ba656275ecb7b5b8f,18.0,11.0
3,268f51b3d2bf41668c1f2f4a9c71a86a,18.0,10.0
4,ca09c05e6c364a6fa5cc567abd45a5e5,18.0,8.0
...,...,...,...
62,45b46d423b6f4f6faf57d9da5592fbe0,1.0,9.0
63,dbfabfe7b3ab40dda63676cd7f157d9e,54.0,10.0
64,fd67ee82d10a4671a23a0ce2be935321,4.0,11.0
65,995a091eda934bdb8be21ea694acc551,3.0,7.0


## B3. Combine two conditions

In [7]:
duckdb.sql("""
    SELECT interview__id, t0_region, nps4_hhsize, nps4_nplots
    FROM households
    WHERE int_result = 1
      AND nps4_hhsize > 6
""").to_df()

,interview__id,t0_region,nps4_hhsize,nps4_nplots
0,2539b62f6d5a4e7080df96ce76b6a2eb,16.0,9.0,2.0
1,9b1bdc92507e495693b487349c520fa2,18.0,7.0,1.0
2,268f51b3d2bf41668c1f2f4a9c71a86a,18.0,10.0,1.0
3,ca09c05e6c364a6fa5cc567abd45a5e5,18.0,8.0,1.0
4,0d923b91b7534e248cbfec4b08bdb6af,16.0,7.0,3.0
...,...,...,...,...
60,45b46d423b6f4f6faf57d9da5592fbe0,1.0,9.0,3.0
61,dbfabfe7b3ab40dda63676cd7f157d9e,54.0,10.0,NaN
62,fd67ee82d10a4671a23a0ce2be935321,4.0,11.0,3.0
63,995a091eda934bdb8be21ea694acc551,3.0,7.0,1.0


---
# Part C — Counting and Summarising

| Function | Result |
|---|---|
| `COUNT(*)` | number of rows |
| `AVG(col)` | mean |
| `SUM(col)` | total |
| `MIN(col)` | smallest value |
| `MAX(col)` | largest value |

## C1. How many completed interviews?

In [8]:
duckdb.sql("""
    SELECT COUNT(*) AS n_complete
    FROM households
    WHERE int_result = 1
""").to_df()

,n_complete
0,756


## C2. Summarise household size

In [9]:
duckdb.sql("""
    SELECT
        AVG(nps4_hhsize) AS mean_size,
        MIN(nps4_hhsize) AS min_size,
        MAX(nps4_hhsize) AS max_size
    FROM households
""").to_df()

,mean_size,min_size,max_size
0,5.034091,1.0,18.0


## C3. Multiple aggregations on filtered data

In [10]:
duckdb.sql("""
    SELECT
        COUNT(*)         AS n_hh,
        AVG(nps4_hhsize) AS mean_size,
        SUM(nps4_nplots) AS total_plots
    FROM households
    WHERE int_result = 1
""").to_df()

,n_hh,mean_size,total_plots
0,756,5.144033,444.0


---
# Part D — Grouping with GROUP BY

```sql
SELECT  group_column, AGG(value_column) AS alias
FROM    table
WHERE   condition
GROUP BY group_column
```

**Rule:** every column in `SELECT` must be either in `GROUP BY` or wrapped in an aggregate.

## D1. Count households per interview result

In [11]:
duckdb.sql("""
    SELECT
        int_result,
        COUNT(*) AS n_hh
    FROM households
    GROUP BY int_result
""").to_df()

,int_result,n_hh
0,NaN,8
1,6.0,1
2,5.0,19
3,7.0,1
4,2.0,2
5,3.0,2
6,1.0,756
7,4.0,2


## D2. Average household size by region

In [12]:
duckdb.sql("""
    SELECT
        t0_region,
        COUNT(*)         AS n_hh,
        AVG(nps4_hhsize) AS mean_size
    FROM households
    WHERE int_result = 1
    GROUP BY t0_region
""").to_df()

,t0_region,n_hh,mean_size
0,16.0,11,5.375000
1,54.0,69,6.650000
2,15.0,13,4.571429
3,52.0,9,NaN
4,10.0,15,5.000000
5,51.0,4,NaN
6,55.0,29,5.333333
7,1.0,34,5.250000
8,19.0,38,NaN
9,24.0,22,6.666667


---
# Part E — Sorting with ORDER BY

```sql
ORDER BY column_name        -- ascending (smallest first, default)
ORDER BY column_name DESC   -- descending (largest first)
```

## E1. Regions with the largest average household size

In [13]:
duckdb.sql("""
    SELECT
        t0_region,
        COUNT(*)         AS n_hh,
        AVG(nps4_hhsize) AS mean_size
    FROM households
    WHERE int_result = 1
    GROUP BY t0_region
    ORDER BY mean_size DESC
""").to_df()

,t0_region,n_hh,mean_size
0,24.0,22,6.666667
1,18.0,35,6.650000
2,54.0,69,6.650000
3,25.0,20,5.857143
4,16.0,11,5.375000
5,55.0,29,5.333333
6,1.0,34,5.250000
7,10.0,15,5.000000
8,5.0,26,4.875000
9,3.0,22,4.571429


## E2. The 5 largest individual households

In [14]:
duckdb.sql("""
    SELECT interview__id, t0_region, nps4_hhsize
    FROM households
    WHERE int_result = 1
    ORDER BY nps4_hhsize DESC
    LIMIT 5
""").to_df()

,interview__id,t0_region,nps4_hhsize
0,6f98f0e8c4944688a1f1e306ac27c898,18.0,18.0
1,7ea5dd31ba46498286328e930642df65,54.0,18.0
2,ec34326c051b4eb3ab8a8c202ac4cb01,24.0,12.0
3,48d37315fc0e403f958205ec3a39cf94,54.0,11.0
4,b4c97cc1bdc344588279566d59a3953f,7.0,11.0


## E3. Regions with the fewest completed interviews

In [15]:
duckdb.sql("""
    SELECT
        t0_region,
        COUNT(*) AS n_hh
    FROM households
    WHERE int_result = 1
    GROUP BY t0_region
    ORDER BY n_hh ASC
""").to_df()

,t0_region,n_hh
0,2.0,3
1,51.0,4
2,22.0,4
3,13.0,7
4,6.0,9
5,23.0,9
6,52.0,9
7,26.0,9
8,16.0,11
9,15.0,13
